# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides an interactive, reproducible guide for loading and exploring the [FAIR^2 Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

## Dataset Source
The dataset source is provided via a Croissant schema URL. All entity references use their Croissant `@id` fields for clarity and reproducibility.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access basic metadata
md = dataset.metadata
print(f"{md.name} (version: {md.version})\n\n{md.description}\n")
print(f"Published on: {md.datePublished}")
print(f"License: {md.license}")
print(f"Identifier: {md.identifier}")

## 2. Data Overview
Review available record sets, their fields, and their Croissant `@id`s.

In [ ]:
# List all record sets and their fields using @id references

print("Record Sets in this dataset:")
record_sets = list(dataset.record_sets)
for record_set in record_sets:
    print(f"- Record set name: '{record_set.name}' @id: '{record_set.id_}'")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id_}) [column: {field.column.id_ if field.column else None}]")
    print("")
if not record_sets:
    print("No record sets found in metadata. This might indicate a package-level dataset or non-standard Croissant usage.")

## 3. Data Extraction
Load data from a specific record set into a pandas DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# If record set information is available, extract all into DataFrames
dfs = {}
record_set_ids = [rs.id_ for rs in dataset.record_sets]

for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        dfs[rs_id] = pd.DataFrame(records)
        print(f"Sample columns for {rs_id}:", dfs[rs_id].columns.tolist())
        display(dfs[rs_id].head())
    else:
        print(f"No records found for {rs_id}.")

if not record_set_ids or not dfs:
    print("No record sets or data were found. Check the Croissant schema for available data objects.")

# Choose the first available record set for downstream analysis
if dfs:
    main_record_set_id = list(dfs.keys())[0]
    main_df = dfs[main_record_set_id]
    print(f"Using record set '{main_record_set_id}' for analysis.")
    print("Fields (columns):", main_df.columns.tolist())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering numeric records, normalizing fields, and grouping by categorical values. Please refer to `@id` for data references.

In [ ]:
# Identify a numeric field by inspecting the available columns
if dfs:
    df = main_df
    # Try to find common MS fields
    candidate_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not candidate_numeric_fields:
        # Try to cast float-looking columns
        float_like = []
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                float_like.append(col)
            except Exception:
                continue
        candidate_numeric_fields = float_like
    # Use the first as numeric
    if candidate_numeric_fields:
        numeric_field = candidate_numeric_fields[0]
        print(f"Using numeric field: '{numeric_field}' for analysis.")
        # Drop obvious outlier values
        thresh = df[numeric_field].mean() + 2 * df[numeric_field].std()
        filtered = df[df[numeric_field] < thresh]
        print(f"Records with {numeric_field} below mean + 2*std ({thresh:.2f}): {len(filtered)} / {len(df)}")

        # Normalization
        filtered[f"{numeric_field}_zscore"] = (
            filtered[numeric_field] - filtered[numeric_field].mean()
        ) / filtered[numeric_field].std()
        display(filtered[[numeric_field, f"{numeric_field}_zscore"]].head())

        # Try grouping by a categorical field
        candidate_group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
        group_field = candidate_group_fields[0] if candidate_group_fields else None
        if group_field:
            group_mean = (
                filtered.groupby(group_field)[numeric_field]
                .mean()
                .sort_values(ascending=False)
            )
            print(f"\nGrouped means for '{numeric_field}' by '{group_field}':\n", group_mean.head())
        else:
            print("No suitable group (categorical) field found for grouping.")
    else:
        print("No numeric fields detected in the record set.")
else:
    print("No DataFrames loaded.")

## 5. Visualization
Visualize the distribution of the selected numeric field and relationships for quick overview.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if dfs and candidate_numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], bins=30, color='skyblue', kde=True)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered)
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion

In this notebook, you:

- Loaded dataset metadata and records using `mlcroissant` directly from a Croissant schema (`@id` referenced throughout).
- Explored the data structure—record sets, fields, and possible columns—all via their semantic identifiers.
- Extracted tabular records into pandas DataFrames for Pythonic analysis.
- Performed basic exploratory analysis: filtered/normalized a mass spectrometry field and grouped data by a biological or annotation field.
- Visualized data distributions to gain further insight into the dataset's structure and content.

For further exploration, see the [https://mlcommons.github.io/croissant/](https://mlcommons.github.io/croissant/) documentation, reference entity `@id`s when extracting or transforming data, and consider submitting additional FAIR-compliant datasets!